# Imbalanced data: five ways to handle it
A synthetic dataset where class 0 is rare. We train logistic regression on it as it is, then try random undersampling,
random oversampling and SMOTE (imbalanced-learn, plus SMOTE written by hand once to see how it works), a balanced
random forest, class weights, and a custom loss in XGBoost. Finally, resampling inside cross-validation with an
imbalanced-learn pipeline.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.datasets import make_classification
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, f1_score, precision_score, recall_score,
                             roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import make_scorer
import xgboost as xgb
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.ensemble import BalancedRandomForestClassifier
from imblearn.pipeline import Pipeline

## 1. The data: 400 rows, two inputs, class 0 is rare

In [ ]:
# make_classification invents a dataset; weights=[0.07] puts about 7% of the rows in class 0
X, y = make_classification(n_samples=400, n_features=2, n_informative=2, n_redundant=0,
                           n_clusters_per_class=2, weights=[0.07], class_sep=0.8, flip_y=0, random_state=59)
# 80% to train, 20% to test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("training rows:", X_train.shape, " test rows:", X_test.shape)
print("training counts per class:", np.bincount(y_train), " test counts:", np.bincount(y_test))

## 2. The problem: high accuracy, the rare class ignored

In [ ]:
# A 'dumb' model: always predict the most frequent class
dumb = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print("dumb model accuracy:", accuracy_score(y_test, dumb.predict(X_test)))

In [ ]:
# One helper to score any model the same way: accuracy, then precision, recall and F1 for the rare class 0, and ROC AUC
def scores(name, y_pred, y_prob=None):
    row = {"model": name,
           "accuracy": accuracy_score(y_test, y_pred),
           "precision (0)": precision_score(y_test, y_pred, pos_label=0, zero_division=0),
           "recall (0)": recall_score(y_test, y_pred, pos_label=0),
           "F1 (0)": f1_score(y_test, y_pred, pos_label=0)}
    if y_prob is not None:
        row["ROC AUC"] = roc_auc_score(y_test, y_prob)
    return row

results = []
base = LogisticRegression().fit(X_train, y_train)
print(classification_report(y_test, base.predict(X_test)))
results.append(scores("logistic regression, as is", base.predict(X_test), base.predict_proba(X_test)[:, 1]))

## 3. Random undersampling: drop majority rows until the classes are equal

In [ ]:
# keep all 21 minority rows and 21 majority rows drawn at random (without replacement)
X_under, y_under = RandomUnderSampler(random_state=42).fit_resample(X_train, y_train)
print(X_train.shape, "->", X_under.shape, " counts:", np.bincount(y_under))

In [ ]:
under = LogisticRegression().fit(X_under, y_under)
results.append(scores("random undersampling", under.predict(X_test), under.predict_proba(X_test)[:, 1]))
pd.DataFrame(results).round(3)

## 4. Random oversampling: copy minority rows until the classes are equal

In [ ]:
# draw minority rows WITH replacement until there are 299 of them, so rows repeat
X_over, y_over = RandomOverSampler(random_state=42).fit_resample(X_train, y_train)
print(X_train.shape, "->", X_over.shape, " counts:", np.bincount(y_over))
# how often each of the 21 minority rows now appears
_, times = np.unique(X_over[y_over == 0], axis=0, return_counts=True)
print("copies per minority row: min", times.min(), " max", times.max())

In [ ]:
over = LogisticRegression().fit(X_over, y_over)
results.append(scores("random oversampling", over.predict(X_test), over.predict_proba(X_test)[:, 1]))
pd.DataFrame(results).round(3)

## 5. SMOTE
First the formula on the worked example: sample (1, 1), neighbour (2, 2), random factor 0.5.

In [ ]:
x, neighbour, lam = np.array([1, 1]), np.array([2, 2]), 0.5
print("new point:", x + lam * (neighbour - x))     # (1.5, 1.5), halfway

In [ ]:
def smote(X, y, k=5, seed=42):
    rng = np.random.default_rng(seed)
    mino = X[y == 0]
    # step 1: the k nearest minority neighbours of every minority point (k + 1 because each point is its own nearest)
    nbrs = NearestNeighbors(n_neighbors=k + 1).fit(mino).kneighbors(mino, return_distance=False)[:, 1:]
    n_new = (y == 1).sum() - len(mino)                # how many new points make the classes equal
    i = rng.integers(len(mino), size=n_new)           # step 2: a random minority point, n_new times
    j = nbrs[i, rng.integers(k, size=n_new)]          # step 3: one of its k neighbours, at random
    lam = rng.random((n_new, 1))                      # step 4: a random factor between 0 and 1
    new = mino[i] + lam * (mino[j] - mino[i])         # step 5: a point on the segment between them
    return np.vstack([X, new]), np.r_[y, np.zeros(n_new, int)]

X_hand, y_hand = smote(X_train, y_train)
print("by hand:", X_train.shape, "->", X_hand.shape, " counts:", np.bincount(y_hand))

In [ ]:
# the same with imbalanced-learn: original rows first, then the 278 synthetic ones
X_smote, y_smote = SMOTE(k_neighbors=5, random_state=42).fit_resample(X_train, y_train)
print("imblearn:", X_train.shape, "->", X_smote.shape, " counts:", np.bincount(y_smote))
# check: every synthetic point lies on a segment between two minority points (here: inside their bounding range)
new = X_smote[len(X_train):]
mino = X_train[y_train == 0]
print("synthetic points inside the minority's range:", bool(((new >= mino.min(0)) & (new <= mino.max(0))).all()))

In [ ]:
sm = LogisticRegression().fit(X_smote, y_smote)
results.append(scores("SMOTE", sm.predict(X_test), sm.predict_proba(X_test)[:, 1]))
pd.DataFrame(results).round(3)

## 6. A balanced random forest
Each tree is trained on a balanced sample: 21 rows of each class, drawn with replacement; the trees vote.

In [ ]:
brf = BalancedRandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
# the defaults: every class is undersampled to 21 rows, drawn with replacement, for each tree
print({k: brf.get_params()[k] for k in ["sampling_strategy", "replacement", "bootstrap"]})
results.append(scores("balanced random forest", brf.predict(X_test), brf.predict_proba(X_test)[:, 1]))

rf = RandomForestClassifier(random_state=42).fit(X_train, y_train)
results.append(scores("random forest, as is", rf.predict(X_test), rf.predict_proba(X_test)[:, 1]))
pd.DataFrame(results).round(3)

## 7. Cost-sensitive learning (a): class weights

In [ ]:
# class 0 mistakes count 5, 25, then 50 times as much as class 1 mistakes
for w in [5, 25, 50]:
    cw = LogisticRegression(class_weight={0: w, 1: 1}).fit(X_train, y_train)
    results.append(scores(f"class_weight {{0: {w}, 1: 1}}", cw.predict(X_test), cw.predict_proba(X_test)[:, 1]))
cw = LogisticRegression(class_weight="balanced").fit(X_train, y_train)
print("weights used by 'balanced':", np.round(len(y_train) / (2 * np.bincount(y_train)), 2))
pd.DataFrame(results).round(3)

Move the slider: the heavier class 0's mistakes, the further the line moves into the majority class.

In [ ]:
gx = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 50)
fig = go.Figure()
for label, colour, name in [(1, "#4C78A8", "class 1 (majority)"), (0, "#E45756", "class 0 (minority)")]:
    m = y_train == label
    fig.add_trace(go.Scatter(x=X_train[m, 0], y=X_train[m, 1], mode="markers", name=name,
                             marker=dict(color=colour, size=8, opacity=0.7)))
weights = [1, 2, 5, 10, 25, 50, 100]
for w in weights:
    m = LogisticRegression(class_weight={0: w, 1: 1}).fit(X_train, y_train)
    (a, b), c = m.coef_[0], m.intercept_[0]
    # the boundary is where a*x1 + b*x2 + c = 0, so x2 = -(a*x1 + c) / b
    fig.add_trace(go.Scatter(x=gx, y=-(a * gx + c) / b, mode="lines", name=f"boundary, weight {w}",
                             line=dict(color="black", width=3), visible=(w == 1)))
steps = [dict(method="update", label=str(w), args=[{"visible": [True, True] + [v == w for v in weights]}])
         for w in weights]
fig.update_layout(sliders=[dict(steps=steps, currentvalue=dict(prefix="class 0 weight: "))],
                  yaxis=dict(range=[X[:, 1].min() - 0.5, X[:, 1].max() + 0.5]), template="simple_white",
                  font=dict(family="Latin Modern Roman"), height=550)
fig

## 7. Cost-sensitive learning (b): a custom loss in XGBoost
A weighted log loss: a mistake on class 0 costs `b`, a mistake on class 1 costs `a`. XGBoost needs its gradient and
Hessian (first and second derivative) with respect to the raw score `z`, for every row.

In [ ]:
def weighted_logloss(a, b):
    def objective(z, dtrain):
        y = dtrain.get_label()
        p = 1 / (1 + np.exp(-z))                       # sigmoid of the raw score
        grad = b * (1 - y) * p - a * y * (1 - p)
        hess = p * (1 - p) * (a * y + b * (1 - y))
        return grad, hess
    return objective

dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test)
for b in [1, 3.5]:
    booster = xgb.train({"max_depth": 3, "eta": 0.1, "base_score": 0.5}, dtrain,
                        num_boost_round=100, obj=weighted_logloss(a=1, b=b))
    # with a custom objective, predict returns raw scores: ask for them and apply the sigmoid ourselves
    p = 1 / (1 + np.exp(-booster.predict(dtest, output_margin=True)))
    results.append(scores(f"XGBoost, custom loss b = {b}", (p > 0.5).astype(int), p))
pd.DataFrame(results).round(3)

## 8. Resampling inside cross-validation
Wrong: resample first, then cross-validate (synthetic copies of a row land in both training and validation folds).
Right: an imbalanced-learn `Pipeline`, which resamples only the training part of each fold.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_minority = make_scorer(f1_score, pos_label=0)          # F1 of class 0

plain = cross_val_score(LogisticRegression(), X_train, y_train, cv=cv, scoring=f1_minority).mean()
pipe = Pipeline([("smote", SMOTE(random_state=42)), ("model", LogisticRegression())])
right = cross_val_score(pipe, X_train, y_train, cv=cv, scoring=f1_minority).mean()
wrong = cross_val_score(LogisticRegression(), X_smote, y_smote, cv=cv, scoring=f1_minority).mean()
print(f"no resampling: {plain:.3f}   SMOTE in a pipeline: {right:.3f}   SMOTE before CV (leaky): {wrong:.3f}")

In [ ]:
# Where does the leaky score come from? Same leaky folds, but score only the real rows of each validation fold
real = np.r_[np.ones(len(y_train), bool), np.zeros(len(y_smote) - len(y_train), bool)]
all_rows, real_rows = [], []
for tr, va in cv.split(X_smote, y_smote):
    pred = LogisticRegression().fit(X_smote[tr], y_smote[tr]).predict(X_smote[va])
    all_rows.append(f1_score(y_smote[va], pred, pos_label=0))
    r = real[va]
    real_rows.append(f1_score(y_smote[va][r], pred[r], pos_label=0))
print(f"leaky, all validation rows: {np.mean(all_rows):.3f}   leaky, real validation rows only: {np.mean(real_rows):.3f}")

## 9. All results side by side

In [ ]:
pd.DataFrame(results).set_index("model").round(3)